# Minimizar y anonimizar: tratar datos personales con cabeza

**Facsímil 9 · Seguridad, privacidad y gobernanza** — capítulo 2 (privacidad y datos personales:
minimización, DPIA y memoria).

Mandar datos personales en crudo a un servicio externo (por ejemplo, a un LLM) es de los errores más
caros —y más multables— que existen. En este cuaderno coges un montón de mensajes de soporte llenos de
datos personales y los preparas para usarlos **sin filtrar a nadie**: detectas la información personal,
la **seudonimizas** de forma coherente, y compruebas con un test de **k-anonimato** si tu tabla deja a
alguien señalado. La idea de fondo, que verás repetida, es **minimizar**: no mover lo que no hace falta.

### Qué vas a aprender
- El principio de **minimización**: la primera y mejor defensa es no tener (ni mover) datos que no
  necesitas.
- A **detectar** datos personales (correos, teléfonos, tarjetas, DNI) con expresiones regulares, y por
  qué ningún detector es perfecto.
- A **seudonimizar** de forma coherente (el mismo dato siempre da el mismo seudónimo, reversible solo con
  una tabla aparte).
- Por qué anonimizar el texto **no basta**: el **k-anonimato** y cómo la generalización protege a los
  casos únicos.

### Cuánto cuesta
Unos 12 minutos. CPU, sin claves.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [1]:
import re
mensajes = [
    "Hola, soy Marta Ruiz, mi telefono es 612345678 y mi correo marta.ruiz@gmail.com",
    "No me funciona la tarjeta 4111 1111 1111 1111, mi DNI es 12345678Z",
    "Llamadme al 699-112-233, pedido de Juan Perez (juan_perez@empresa.es)",
]
print("Mensajes en crudo (NO se pueden mandar asi a ningun servicio externo):")
for m in mensajes: print("  -", m)


Mensajes en crudo (NO se pueden mandar asi a ningun servicio externo):
  - Hola, soy Marta Ruiz, mi telefono es 612345678 y mi correo marta.ruiz@gmail.com
  - No me funciona la tarjeta 4111 1111 1111 1111, mi DNI es 12345678Z
  - Llamadme al 699-112-233, pedido de Juan Perez (juan_perez@empresa.es)


## 1. Minimizar: la defensa que no cuesta nada

Antes de cualquier técnica, la pregunta clave es: **¿necesito de verdad estos datos para la tarea?** Si
quieres clasificar el *tema* de un ticket, no necesitas el nombre ni el teléfono del cliente. La
minimización —no recoger, no mover, no almacenar lo que no hace falta— elimina el riesgo de raíz: lo que
no tienes, no se puede filtrar. Todo lo que viene después (detectar, seudonimizar) es para lo que **sí**
necesitas tratar.


## 2. Detectar lo personal

Definimos patrones para lo más habitual: correos, teléfonos, tarjetas y DNI. No es perfecto (ningún
detector lo es: un teléfono escrito con letras se escaparía), pero atrapa la mayor parte y deja claro
qué hay que proteger.


In [2]:
PATRONES = {
    "EMAIL":    r"[\w.\-]+@[\w\-]+\.[\w.\-]+",
    "TARJETA":  r"\b(?:\d[ -]?){13,16}\b",
    "DNI":      r"\b\d{8}[A-Za-z]\b",
    "TELEFONO": r"\b\d{3}[ -]?\d{3}[ -]?\d{3}\b",
}
def detecta(texto):
    out = []
    for tipo, patron in PATRONES.items():
        for m in re.finditer(patron, texto):
            out.append((tipo, m.group().strip()))
    return out

for m in mensajes:
    print(m[:42], "...")
    for tipo, valor in detecta(m): print(f"     [{tipo}] {valor}")


Hola, soy Marta Ruiz, mi telefono es 61234 ...
     [EMAIL] marta.ruiz@gmail.com
     [TELEFONO] 612345678
No me funciona la tarjeta 4111 1111 1111 1 ...
     [TARJETA] 4111 1111 1111 1111
     [DNI] 12345678Z
Llamadme al 699-112-233, pedido de Juan Pe ...
     [EMAIL] juan_perez@empresa.es
     [TELEFONO] 699-112-233


## 3. Seudonimizar de forma coherente

No basta con tachar: a veces necesitas saber que dos mensajes son de la **misma** persona sin saber
quién es. Sustituimos cada dato por una etiqueta **estable** (el mismo correo siempre da el mismo
`EMAIL_1`). Es reversible solo con la tabla de equivalencias, que guardas **aparte y cifrada**. Esto es
seudonimización (reversible con la clave), distinta de la anonimización (irreversible).


In [3]:
mapa = {}; contador = {}
def seudonimo(tipo, valor):
    if valor not in mapa:
        contador[tipo] = contador.get(tipo, 0) + 1
        mapa[valor] = f"{tipo}_{contador[tipo]}"
    return mapa[valor]
def anonimiza(texto):
    out = texto
    for tipo, patron in PATRONES.items():
        out = re.sub(patron, lambda m: seudonimo(tipo, m.group().strip()), out)
    return out

print("Mensajes listos para procesar (sin datos personales en claro):\n")
for m in mensajes: print("  ", anonimiza(m))
print(f"\nTabla de equivalencias (se guarda APARTE, cifrada): {len(mapa)} entradas")


Mensajes listos para procesar (sin datos personales en claro):

   Hola, soy Marta Ruiz, mi telefono es TELEFONO_1 y mi correo EMAIL_1
   No me funciona la tarjeta TARJETA_1, mi DNI es DNI_1
   Llamadme al TELEFONO_2, pedido de Juan Perez (EMAIL_2)

Tabla de equivalencias (se guarda APARTE, cifrada): 6 entradas


## 4. ¿Tu tabla deja a alguien señalado? k-anonimato

Anonimizar el texto no basta si la **combinación** de columnas identifica a una persona. El
**k-anonimato** exige que cada combinación de atributos «cuasi-identificadores» (código postal + edad +
género, por ejemplo) se repita en al menos *k* personas. Si alguien es **único** por su combinación, está
señalado: aunque le hayas borrado el nombre, se puede reidentificar cruzando con otra fuente.


In [4]:
from collections import Counter
tabla = [
    {"cp": "28001", "edad": 34, "genero": "F"},
    {"cp": "28001", "edad": 34, "genero": "F"},
    {"cp": "28001", "edad": 34, "genero": "F"},
    {"cp": "08025", "edad": 51, "genero": "M"},   # unico en su combinacion
    {"cp": "28001", "edad": 34, "genero": "F"},
]
combos = Counter((r["cp"], r["edad"], r["genero"]) for r in tabla)
k = min(combos.values())
print("Tamano de cada grupo de cuasi-identificadores:", dict(combos))
print(f"k del conjunto = {k}", "-> ALGUIEN es unico y reidentificable!" if k < 2 else "-> ok")


Tamano de cada grupo de cuasi-identificadores: {('28001', 34, 'F'): 4, ('08025', 51, 'M'): 1}
k del conjunto = 1 -> ALGUIEN es unico y reidentificable!


## 5. La defensa: generalizar para subir la k

Para proteger a los casos únicos, **generalizamos**: en vez del código postal completo, sus 3 primeras
cifras; en vez de la edad exacta, un tramo de 10 años. Así varias personas comparten la misma
combinación y la k sube. El precio es **perder detalle**: proteger cuesta resolución, y es una decisión
consciente. Lo medimos.


In [5]:
def generaliza(r):
    return (r["cp"][:3], (r["edad"]//10)*10, r["genero"])   # CP a 3 cifras, edad por decenas

tabla2 = [
    {"cp":"28001","edad":34,"genero":"F"}, {"cp":"28004","edad":37,"genero":"F"},
    {"cp":"28010","edad":31,"genero":"F"}, {"cp":"08025","edad":51,"genero":"M"},
    {"cp":"08021","edad":55,"genero":"M"}, {"cp":"08029","edad":58,"genero":"M"},
]
k_antes = min(Counter((r["cp"], r["edad"], r["genero"]) for r in tabla2).values())
k_despues = min(Counter(generaliza(r) for r in tabla2).values())
print(f"k SIN generalizar:  {k_antes}  (cada uno es unico -> reidentificable)")
print(f"k generalizando:    {k_despues}  (varios comparten combinacion -> protegidos)")
print("\nGeneralizar sube la k y protege a los unicos, a costa de perder detalle. Es un trade-off consciente.")


k SIN generalizar:  1  (cada uno es unico -> reidentificable)
k generalizando:    3  (varios comparten combinacion -> protegidos)

Generalizar sube la k y protege a los unicos, a costa de perder detalle. Es un trade-off consciente.


## 6. Pruébalo tú

1. **Añade un patrón de IBAN** y un mensaje que lo contenga. ¿Qué otros datos personales se te ocurren
   que habría que cubrir (matrículas, número de la Seguridad Social)?
2. **Rompe el detector:** escribe un teléfono como «seis uno dos...». Los regex no lo pillan; por eso en
   serio se combinan con modelos de detección. Ningún filtro es perfecto.
3. **Sube el nivel de generalización** (edad por tramos de 20, CP a 2 cifras): ¿llegas a k≥3 para todos?
   ¿Cuánto detalle has sacrificado?
4. **l-diversidad:** investiga por qué k-anonimato no basta si todos los de un grupo comparten el mismo
   valor sensible (p. ej. la misma enfermedad).


## 7. Errores comunes

- **Saltarse la minimización.** La defensa más barata es no tener el dato. Pregúntate siempre si lo
  necesitas.
- **Confiar el 100% en los regex.** Se escapan formatos raros. Combínalos con modelos y asume que algo
  pasará.
- **Anonimizar el texto y olvidar la combinación de columnas.** El k-anonimato vigila justo eso: que la
  suma de atributos no señale a nadie.
- **Confundir seudonimizar y anonimizar.** Seudonimizar es reversible con una clave (que hay que
  proteger); anonimizar, irreversible. No son lo mismo a efectos legales.


## 8. Qué te llevas

- **Minimizar** (no mover lo que no hace falta) es la primera defensa; **seudonimizar** de forma coherente
  te deja trabajar sin exponer identidades.
- Ningún detector es perfecto: se combinan reglas y modelos, y se asume que algo se escapará.
- Anonimizar el texto no basta: el **k-anonimato** vigila que la *combinación* de datos no señale a nadie;
  **generalizar** sube la k a costa de detalle. Proteger es un trade-off consciente.

**Para seguir:** el capítulo 3 trata la seguridad de las aplicaciones LLM (el siguiente cuaderno: inyección
de prompt); el capítulo 4, el cumplimiento y los paquetes de evidencia.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*